**4.1 · Conteos y estado de los indicadores**

In [0]:
%sql
SELECT 'bronze.vuelos' AS tabla, COUNT(*) AS registros FROM aerolinea.bronze.vuelos
UNION ALL SELECT 'silver.vuelos', COUNT(*) FROM aerolinea.silver.vuelos
UNION ALL SELECT 'gold.vuelos_enriquecidos', COUNT(*) FROM aerolinea.gold.vuelos_enriquecidos
UNION ALL SELECT 'gold.mantenimientos_enriquecidos', COUNT(*) FROM aerolinea.gold.mantenimientos_enriquecidos
UNION ALL SELECT 'gold.base_ml_mantenimiento', COUNT(*) FROM aerolinea.gold.base_ml_mantenimiento

tabla,registros
bronze.vuelos,100
silver.vuelos,100
gold.vuelos_enriquecidos,100
gold.mantenimientos_enriquecidos,100
gold.base_ml_mantenimiento,100


In [0]:
vuelos = spark.table("aerolinea.gold.vuelos_enriquecidos").count()
assert vuelos == 100, "El conteo no coincide con la instantánea de clase."
assert spark.table("aerolinea.gold.mantenimientos_enriquecidos").count() == 100
assert spark.table("aerolinea.gold.base_ml_mantenimiento").count() == 100

**4.2 · Reconciliación de conteos y costos**

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW control_reconciliacion AS
SELECT 'vuelos_por_grupo' AS regla,
       (SELECT COUNT(*) FROM aerolinea.silver.vuelos) AS esperado,
       (SELECT SUM(total_vuelos) FROM aerolinea.gold.kpi_puntualidad_vuelos) AS obtenido
UNION ALL
SELECT 'mantenimientos_por_grupo',
       (SELECT COUNT(*) FROM aerolinea.silver.mantenimientos),
       (SELECT SUM(total_mantenimientos) FROM aerolinea.gold.kpi_costo_mantenimiento)
UNION ALL
SELECT 'vuelos_por_aeronave_mes',
       (SELECT COUNT(*) FROM aerolinea.silver.vuelos),
       (SELECT SUM(total_vuelos) FROM aerolinea.gold.resumen_aeronave_mes)
UNION ALL
SELECT 'mantenimientos_por_aeronave_mes',
       (SELECT COUNT(*) FROM aerolinea.silver.mantenimientos),
       (SELECT SUM(total_mantenimientos) FROM aerolinea.gold.resumen_aeronave_mes)
UNION ALL
SELECT 'costo_por_grupo',
       (SELECT SUM(costo_usd) FROM aerolinea.silver.mantenimientos),
       (SELECT SUM(costo_total_usd) FROM aerolinea.gold.kpi_costo_mantenimiento)
UNION ALL
SELECT 'costo_por_aeronave_mes',
       (SELECT SUM(costo_usd) FROM aerolinea.silver.mantenimientos),
       (SELECT SUM(costo_total_usd) FROM aerolinea.gold.resumen_aeronave_mes)

In [0]:
reconciliacion = spark.table("control_reconciliacion")
display(reconciliacion)
assert reconciliacion.filter("obtenido IS NULL OR esperado <> obtenido").count() == 0, "Un resumen no coincide con el detalle."

regla,esperado,obtenido
vuelos_por_grupo,100.00,100.00
mantenimientos_por_grupo,100.00,100.00
vuelos_por_aeronave_mes,100.00,100.00
mantenimientos_por_aeronave_mes,100.00,100.00
costo_por_grupo,2589355.38,2589355.38
costo_por_aeronave_mes,2589355.38,2589355.38


**4.3 · Revisión de las tasas**

In [0]:
%sql
SELECT SUM(total_vuelos) AS total,
       SUM(vuelos_a_tiempo) AS a_tiempo, SUM(vuelos_retrasados) AS retrasados,
       SUM(vuelos_cancelados) AS cancelados, SUM(vuelos_operados) AS operados,
       SUM(rutas_por_revisar) AS rutas_por_revisar,
       ROUND(100.0 * SUM(vuelos_a_tiempo) / NULLIF(SUM(vuelos_operados), 0), 2) AS puntualidad_pct
FROM aerolinea.gold.kpi_puntualidad_vuelos

total,a_tiempo,retrasados,cancelados,operados,rutas_por_revisar,puntualidad_pct
100,40,29,31,69,25,57.97


In [0]:
estados = spark.sql("""
SELECT SUM(total_vuelos) AS total, SUM(vuelos_operados) AS operados,
       SUM(vuelos_a_tiempo) AS a_tiempo, SUM(vuelos_retrasados) AS retrasados,
       SUM(vuelos_cancelados) AS cancelados
FROM aerolinea.gold.kpi_puntualidad_vuelos
""").first()
assert estados["a_tiempo"] + estados["retrasados"] == estados["operados"]
assert estados["operados"] + estados["cancelados"] == estados["total"]

**4.4 · Consulta, plan, archivos e historial**

In [0]:
%sql
SELECT origen_pais, COUNT(*) AS vuelos_operados
FROM aerolinea.gold.vuelos_enriquecidos
WHERE operado
GROUP BY origen_pais

origen_pais,vuelos_operados
Francia,13
Reino Unido,15
España,32
EEUU,9


In [0]:
%sql
EXPLAIN FORMATTED
SELECT origen_pais, COUNT(*) AS operados
FROM aerolinea.gold.vuelos_enriquecidos
WHERE operado
GROUP BY origen_pais

plan
"== Physical Plan == AdaptiveSparkPlan (6) +- == Initial Plan == PhotonResultStage (5) +- PhotonColumnarToRow (4) +- PhotonGroupingAgg (3) +- PhotonProject (2) +- PhotonScan parquet aerolinea.gold.vuelos_enriquecidos (1) (1) PhotonScan parquet aerolinea.gold.vuelos_enriquecidos Output [2]: [origen_pais#26692, operado#26699] DictionaryFilters: [operado#26699] Location: PreparedDeltaFileIndex [s3://dbstorage-prod-4hjpqwpo5t/uc/aa8582a0-dd24-4de9-8a7f-eae236b261d5/475e2037-1b01-4cd4-a8a5-0941c133b513/__unitystorage/catalogs/73493fc4-731a-4df5-b71c-dfb6ef011350/tables/d1abd6f1-3d0c-4494-a114-4604a8c7fbe2] ReadSchema: struct RequiredDataFilters: [isnotnull(operado#26699), operado#26699] (2) PhotonProject Input [2]: [origen_pais#26692, operado#26699] Arguments: [origen_pais#26692] (3) PhotonGroupingAgg Input [1]: [origen_pais#26692] Arguments: [origen_pais#26692], [count(1)], [count(1)#26701L], [origen_pais#26692, count(1)#26701L AS operados#26653L], true (4) PhotonColumnarToRow Input [2]: [origen_pais#26692, operados#26653L] (5) PhotonResultStage Input [2]: [origen_pais#26692, operados#26653L] (6) AdaptiveSparkPlan Output [2]: [origen_pais#26692, operados#26653L] Arguments: isFinalPlan=false == Photon Explanation == The query is fully supported by Photon. == Optimizer Statistics (table names per statistics state) == missing = partial = full = vuelos_enriquecidos"


In [0]:
%sql
DESCRIBE DETAIL aerolinea.gold.vuelos_enriquecidos

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,28a6094f-071b-49a1-92c7-4cb52e3d7d12,aerolinea.gold.vuelos_enriquecidos,null,,2026-10-24T19:11:45.449Z,2026-10-24T19:11:51.000Z,List(),List(),1,7388,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, delta.enableRowTracking -> true, delta.checkpointPolicy -> v2, io.unitycatalog.tableId -> d1abd6f1-3d0c-4494-a114-4604a8c7fbe2, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-11001b5b-a1c9-4dec-8761-e91f570fbcad, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-d3e6b8dd-0e11-4090-9481-7834c0525110)",3,7,"List(appendOnly, deletionVectors, domainMetadata, invariants, rowTracking, v2Checkpoint)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
DESCRIBE HISTORY aerolinea.gold.vuelos_enriquecidos

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-10-24T19:11:51.000Z,72914905396047,andyduqueep@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""databricks.internal.autoUpgrades.delta.enableRowTracking"":""true"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""databricks.internal.autoUpgrades.delta.checkpointPolicy"":""v2"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""d1abd6f1-3d0c-4494-a114-4604a8c7fbe2"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-11001b5b-a1c9-4dec-8761-e91f570fbcad"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-d3e6b8dd-0e11-4090-9481-7834c0525110""}, statsOnLoad -> true)",null,List(2453944625233349),a30193d0-2da6-4741-b1b4-84cb1c1958a7,1024-190651-en1932tp-v2n,1,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 7388, numDeletionVectorsRemoved -> 0, numOutputRows -> 100, numOutputBytes -> 7388)",null,Databricks-Runtime/19.11.x-aarch64-photon-scala2.13
1,2026-10-24T19:08:27.000Z,72914905396047,andyduqueep@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""databricks.internal.autoUpgrades.delta.enableRowTracking"":""true"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""databricks.internal.autoUpgrades.delta.checkpointPolicy"":""v2"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""d1abd6f1-3d0c-4494-a114-4604a8c7fbe2"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-11001b5b-a1c9-4dec-8761-e91f570fbcad"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-d3e6b8dd-0e11-4090-9481-7834c0525110""}, statsOnLoad -> true)",null,List(2453944625233349),a490ad76-eb3a-471d-8b0e-11a701c202c6,1024-190651-en1932tp-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 7720, numDeletionVectorsRemoved -> 0, numOutputRows -> 100, numOutputBytes -> 7388)",null,Databricks-Runtime/19.11.x-aarch64-photon-scala2.13
0,2026-10-24T19:07:35.000Z,72914905396047,andyduqueep@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""databricks.internal.autoUpgrades.delta.enableRowTracking"":""true"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""databricks.internal.autoUpgrades.delta.checkpointPolicy"":""v2"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""d1abd6f1-3d0c-4494-a114-4604a8c7fbe2"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-11001b5b-a1c9-4dec-8761-e91f570fbcad"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-d3e6b8dd-0e11-4090-9481-7834c0525110""}, statsOnLoad -> true)",null,List(2453944625233349),a3fe3ac6-aba5-4fed-8373-64a97818b993,1024-190651-en1932tp-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 100, numOutputBytes -> 7720)",null,Databricks-Runtime/19.11.x-aarch64-photon-scala2.13
